In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 02. Single-Band (C-band) Baseline\n",
    "Establishing the baseline performance of standard 64-QAM in the C-band over 80 km SSMF.\n",
    "\n",
    "**Goal:** Measure baseline GMI, SNR, and BER without multi-band interference or neural equalization. This acts as the comparison point for the 485 Tb/s multi-band claim."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "sys.path.append('..')\n",
    "import yaml\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "from src.modulation import build_band_plan, generate_pcs_symbols\n",
    "from src.fiber_channel import run_ssfm\n",
    "from src.amplifiers import apply_amplifiers\n",
    "from src.dsp import receiver_dsp\n",
    "from src.metrics import compute_metrics\n",
    "\n",
    "with open('../config/simulation_params.yaml', 'r') as f:\n",
    "    cfg = yaml.safe_load(f)\n",
    "\n",
    "# Isolate C-band only\n",
    "c_band_cfg = {'C': cfg['bands']['C']}\n",
    "c_agg_cfg = {'total_channels': cfg['bands']['C']['channels'], 'total_bandwidth_THz': cfg['bands']['C']['bandwidth_THz']}\n",
    "\n",
    "band_plan = build_band_plan(c_band_cfg, c_agg_cfg)\n",
    "\n",
    "# Force uniform 64-QAM for baseline (disable PCS shaping)\n",
    "mod_cfg = cfg['modulation'].copy()\n",
    "mod_cfg['pcs_learning_method'] = 'none'\n",
    "\n",
    "tx = generate_pcs_symbols(band_plan, mod_cfg, cfg['launch_power'], samples=32768)\n",
    "print(f\"Transmitting {tx['n_channels']} channels in C-band...\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Run Pipeline\n",
    "rx_raw = run_ssfm(tx, cfg['fiber'], cfg['ssfm'])\n",
    "rx_amp = apply_amplifiers(rx_raw, c_band_cfg)\n",
    "rx_dsp = receiver_dsp(rx_amp, cfg['dsp'], cfg['fiber'])\n",
    "\n",
    "# Compute Metrics\n",
    "metrics = compute_metrics(rx_dsp, tx, band_plan, cfg['metrics'])\n",
    "\n",
    "print(f\"C-band Aggregate Capacity: {metrics['aggregate_capacity_Tbs']:.2f} Tb/s\")\n",
    "print(f\"Mean GMI: {metrics['gmi_mean']:.2f} bits/symbol\")\n",
    "print(f\"Mean SNR: {metrics['snr_mean_dB']:.2f} dB\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Plot baseline SNR per channel\n",
    "plt.figure(figsize=(10, 4))\n",
    "plt.plot(metrics['snr_per_channel_dB'], marker='o', ls='-', label='C-band SNR')\n",
    "plt.title('Baseline SNR per Channel (C-Band Only)')\n",
    "plt.xlabel('Channel Index')\n",
    "plt.ylabel('SNR (dB)')\n",
    "plt.grid(True)\n",
    "plt.legend()\n",
    "plt.show()"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.10"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}